# FallVision: the complete pipeline in one Kaggle notebook

Person collapse detection (DS5602 course project). This notebook runs **everything after the video
download**: pose caching (YOLO11m-pose + ByteTrack), feature extraction, 2-s windows, training of the
final GRU and the comparison models, learning curves, model choice on validation, and the final
evaluation of the submitted frame-by-frame system with the threshold re-chosen on validation.

Nothing is implemented here: every stage is a script in `src/` called through `run_pipeline.py`, the
same code that runs on a laptop (CPU or GPU). Kaggle-specific things (where the inputs are, pushing
results) are in the settings cell and `src/kaggle_helpers.py` only.

**What you need to do**
1. *Add Input*: the videos. Either the outputs of `notebooks/01_download_omnifall_videos` and
   `02_download_gmdcsa24`, or the packed `FallVision_dataset` (notebook 06), or your own dataset with the
   same layout (`docs/DATA.md`). Optional: the private `fallvision-own-videos` dataset.
2. *Settings*: Accelerator **GPU T4 x1**, Internet **on**. *Add-ons > Secrets*: `GITHUB_TOKEN`
   (only if `PUSH_RESULTS = True`).
3. **Save Version > Save & Run All (Commit)**. Total time on a T4: about 9-10 h (poses 6-7 h, training
   30 min, evaluation 2 h). Kaggle stops sessions after 12 h; if that is a risk, run the three staged
   notebooks `notebooks/03`, `04`, `05` instead (same code, same results).

**What you get** (this notebook's Output): `export/` with `models/gru_hard.pt`, `config.yaml` (the
threshold chosen on validation), `results/` (all tables, `training_curves.png`) and `export.zip`;
plus `work/` with every intermediate file.

In [ ]:
# ---- settings: the only cell you need to change ----
REPO_URL = "https://github.com/Prish1509/project_fallvision.git"   # the code (public repository)
BRANCH = "main"

# WHERE THE VIDEOS ARE (one place). Leave DATA = "" to find them automatically among the notebook's inputs
# (the outputs of notebooks 01/02, or the packed FallVision_dataset of notebook 06). Or give the folder that
# holds omnifall/ and GMDCSA24/ (and own/), e.g. "/kaggle/input/datasets/<user>/fallvision-dataset".
DATA = ""

STAGES = ["poses", "features", "windows", "train", "choose", "evaluate"]   # remove stages to run a part
ALL_MODELS = True               # also train the comparison models (GRU w/o hard negatives, w/o augmentation, RF)
PUSH_RESULTS = True             # push results/ and config.yaml to the repository (needs the GITHUB_TOKEN secret)
POSES_MAX_HOURS = 7.5           # pose caching stops cleanly after this (Kaggle sessions end after 12 h)

REPO = "/kaggle/working/project_fallvision"     # code
WORK = "/kaggle/working/work"           # pose caches, features, windows, probabilities, scores (kept in the Output)
EXPORT = "/kaggle/working/export"       # trained model + config + results, ready to download

## 1. Code and packages

In [ ]:
import os, shutil, subprocess, sys
os.makedirs("/kaggle/working", exist_ok=True)
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, os.path.join(REPO, "src"))
import kaggle_helpers as kh

# the exact versions from requirements.txt of the packages Kaggle does not have (torch is Kaggle's own)
pins = [l.split("#")[0].strip() for l in open("requirements.txt")
        if l.split("==")[0].strip() in ("ultralytics", "ultralytics-thop", "lap")]
kh.run([sys.executable, "-m", "pip", "install", "-q", *pins])

import torch
print("code version:", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - turn it on: Settings > Accelerator > GPU T4 x1")

## 2. Inputs: videos and the pretrained pose model

In [ ]:
if DATA:
    OMNI, GMD = os.path.join(DATA, "omnifall"), os.path.join(DATA, "GMDCSA24")
    OWN = os.path.join(DATA, "own") if os.path.isdir(os.path.join(DATA, "own")) else None
    print("videos from DATA:", DATA)
else:
    OMNI, GMD = kh.find_omnifall(), kh.find_gmd()
    OWN = kh.find_own(skip=(OMNI, GMD))
assert OMNI and GMD and os.path.isdir(OMNI) and os.path.isdir(GMD), "video folders not found: see the top of this notebook"
kh.run([sys.executable, "src/check_dataset.py", "--omnifall-root", OMNI, "--gmd-root", GMD, *(["--own-root", OWN] if OWN else [])])
kh.get_yolo("models/yolo11m-pose.pt")
kh.env_report("results/env_kaggle.txt")

## 3. The pipeline, stage by stage

In [ ]:
import csv, time
t_all = time.time()
for stage in STAGES:
    cmd = [sys.executable, "run_pipeline.py", "--omnifall-root", OMNI, "--gmd-root", GMD, "--work", WORK,
           "--device", "cuda:0", "--max-hours", str(POSES_MAX_HOURS), "--from", stage, "--to", stage]
    if OWN:
        cmd += ["--own-root", OWN]
    if ALL_MODELS:
        cmd += ["--all-models"]
    print(f"\n{'#' * 100}\n# stage: {stage}\n{'#' * 100}", flush=True)
    kh.run(cmd)
    if stage == "poses":
        index = list(csv.DictReader(open(f"{WORK}/poses/index.csv")))
        bad = [r for r in index if r["status"] not in ("cached", "already")]
        skipped = [r for r in bad if r["status"].startswith("skipped")]
        print(f"poses: {len(index) - len(bad)} of {len(index)} videos cached; problems: {len(bad)}")
        if skipped:
            raise SystemExit(f"{len(skipped)} videos were not cached before POSES_MAX_HOURS: training on a partial "
                             "dataset would give a weaker model, so this notebook stops here. Use the staged notebooks "
                             "(notebooks/03, 04, 05: same code) or raise POSES_MAX_HOURS.")
print(f"\nAll stages done in {(time.time() - t_all) / 3600:.1f} h")

## 4. Results of this run next to the committed ones

In [ ]:
from IPython.display import Image, display
if os.path.exists("results/training_curves.png"):
    display(Image("results/training_curves.png"))
    print(open("results/training_curves.md").read())
if os.path.exists(f"{WORK}/causal_scores.csv"):
    shutil.copyfile(f"{WORK}/causal_scores.csv", "results/causal_scores.csv")
    print("Final system, chosen setting per split, committed version -> this run:\n")
    kh.compare_eval(kh.git_text(REPO, "results/causal_eval.md"), open("results/causal_eval.md").read())
    print()
    kh.show_change(REPO, "config.yaml")
if os.path.exists("results/fusion.md"):
    kh.show_change(REPO, "results/fusion.md")

## 5. Export: trained model, config and results

`export/` holds what the submission package needs from this run; `export.zip` is the same as one file.
Download it from the **Output** tab (or with the Kaggle CLI: `kaggle kernels output <user>/<notebook> -p DIR`),
then put `models/gru_hard.pt` into the project's `models/` folder and `config.yaml` at its root.

In [ ]:
shutil.rmtree(EXPORT, ignore_errors=True)
os.makedirs(f"{EXPORT}/models", exist_ok=True)
for src, dst in (("models/gru_hard.pt", "models/gru_hard.pt"), ("models/gru_hard_last.pt", "models/gru_hard_last.pt"),
                 ("config.yaml", "config.yaml")):
    if os.path.exists(src):
        shutil.copyfile(src, f"{EXPORT}/{dst}")
shutil.copytree("results", f"{EXPORT}/results", dirs_exist_ok=True)
shutil.make_archive(EXPORT, "zip", EXPORT)
for f in sorted(os.listdir(f"{EXPORT}/models")) + ["config.yaml"]:
    p = os.path.join(EXPORT, "models", f) if f.endswith(".pt") else os.path.join(EXPORT, f)
    print(f"export/{'models/' if f.endswith('.pt') else ''}{f}  ({os.path.getsize(p) / 1e6:.1f} MB)")
print(f"export/results/: {sum(len(fs) for _, _, fs in os.walk(EXPORT + '/results'))} files;  "
      f"export.zip: {os.path.getsize(EXPORT + '.zip') / 1e6:.1f} MB")

## 6. Save the results to the repository (the model itself is never put in git)

In [ ]:
if PUSH_RESULTS:
    kh.push(REPO, ["results", "config.yaml"], "Kaggle pipeline run: training, curves, model choice, final evaluation",
            REPO_URL, BRANCH)
else:
    print("PUSH_RESULTS is False: nothing pushed; everything is in the Output tab.")